# 33 - Public pretrained DMD: frozen feasibility, NOT training

Run02 after a runtime reset and04 on this runtime for verified NBIS. The completed29 artifacts are
read only and remain FAILED. Do NOT rerun29 or start30/31. This notebook downloads179,093,287 bytes
of PUBLIC author-provided DMD weights; no fingerprint images leave this runtime/your Drive.
Source checksums use canonical Git blob bytes, not Windows checkout line-ending conversions.
No new Python packages, VeriFinger, fptools, custom CUDA extensions, optimizer or training.

**Scope fixed before seeing33 scores:** use original29 calibration/gate identities inside main TRAIN,
first3 byte-distinct impressions per identity, same-DB nonmated pairs only. Those groups have already
been inspected in29/32: this is DEVELOPMENT feasibility, not a fresh independent benchmark.
Official44.6M-parameter DMD, SD14 weight label,128px/500ppi, original local masked dense cosine and
LSA-R formulas. Replace only the assignment solver with SciPy CPU Hungarian, with numerical tests.
Retain at most24 highest-quality original-image NBIS minutiae (ties x/y/theta); require at least4.
Anchors are extracted once from ORIGINAL TRAIN images and reused unchanged across dry-like conditions.
This tests fixed-reference-anchor supervision, NOT deployment detection on degraded inputs.

NBIS5 `.xyt`: bottom-left origin, y=height-native_y, degrees counterclockwise from east, pointing
out of ridge endings. DMD: image coordinates, clockwise from east. Convert `(x,y,t)` to
`(x,height-y,(-t)%360)`; do NOT add an unexplained180-degree correction. Geometric conversion and
cardinal directions are checked. Equality of NBIS and VeriFinger's semantic minutia directions is
NOT attested. Pure tensor bilinear sampling is the affine equivalent of upstream zero-flow TPS
(output center64, border127.5/255); not bit-exact OpenCV/TPS quantization.

Quality criteria retain29's .80 clean AUC/.70 dry AUC/.25 clean TAR/.10 dry TAR, with a predeclared
90% pair-matchability floor. Recalibrate THIS model on clean calibration impostors at1% conditional FMR;
never reuse29 cosine thresholds. Report conditional FMR and attempted-pair TAR separately; missing
features are UNMATCHABLE, never fabricated zero scores. No validation/test/FVC2004 images opened.
Even a pass DOES NOT authorize30/31: full LSA-R has discrete assignments/ranking and is not end-to-end
differentiable. A separate input-gradient test covers only fixed-anchor sampling, DMD and local dense score.
The official local denominator floor can make self-scores below1 for near-zero masked feature energy;
do not silently remove this floor or reinterpret synthetic-noise fixture scores as fingerprint accuracy.

**Compute:** CPU supported, but full288-image/4-condition evaluation can be slow. A T4 is expected to
be enough for batch8 inference; A100 is unnecessary. Gradient check always CPU because CUDA grid_sample
backward is not deterministic. Actual CUDA memory/time is NOT locally certified. Per-image checksummed
feature/anchor caches survive interruptions; unexpected errors stop with explicit logs.

**Provenance/license limits:** official README labels DMD weights SD14-trained; paper also uses DPF masks
for augmentation. This is not a complete weight-level data attestation or proof of identity nonoverlap.
Upstream LICENCE is Apache2, while README restricts use to noncommercial academic research. Preserve
both notices; do not assume unrestricted redistribution or publishable reproducible training provenance.
No novelty or journal-level improvement is established by this feasibility check.


In [ ]:
import os
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
import hashlib
import importlib.util
import inspect
import json
import math
import shutil
import sys
import time
import types
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch.utils.data import DataLoader, Subset
from google.colab import drive
DEVICE = torch.device('cpu')
torch.set_num_threads(min(4, os.cpu_count() or 1))
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
torch.use_deterministic_algorithms(True)
drive.mount('/content/drive')
PROJECT_ROOT = Path('/content/drive/MyDrive/FingerprintResearch')
LOCAL_ROOT = Path('/content/fingerprint_research')
DATA_ROOT = LOCAL_ROOT / 'data'
SPLIT_DIR = PROJECT_ROOT / 'protocols' / 'fvc_split_v1'
PAIR_DIR = PROJECT_ROOT / 'protocols' / 'nbis_validation_pairs_v1'
SIM_DIR = PROJECT_ROOT / 'protocols' / 'structured_degradation_v1'
M0_DIR = PROJECT_ROOT / 'experiments' / 'm0_unet_v1_seed20261110'
M1_DIR = PROJECT_ROOT / 'experiments' / 'm1_shared_curriculum_v1_seed20261110' / 'dd1427cfad6621c577bf'
SEED = 20261110
RUN_SEEDS = {'dry_l1_seed20261110': 20261110}
RUN_NAMES = tuple(RUN_SEEDS)
M2_DIR = PROJECT_ROOT / 'experiments' / 'm2_structure_loss_v1_seed20261110' / '92579202157d85254b7f'
SPECIALIZATION_ROOT = PROJECT_ROOT / 'experiments' / 'm5_dry_specialization_v1'
LAMBDA_GRADIENT = 0.1
IDENTITY_FRACTION = 0.1

def file_digest(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()

def json_bytes(value):
    return (json.dumps(value, indent=2, sort_keys=True, allow_nan=False) + '\n').encode('utf-8')

def write_verified(path, data):
    pending = path.with_suffix(path.suffix + '.part')
    pending.write_bytes(data)
    if pending.read_bytes() != data:
        raise RuntimeError(f'Write verification failed: {path}')
    pending.replace(path)

def write_csv(path, frame):
    write_verified(path, frame.to_csv(index=False, lineterminator='\n').encode('utf-8'))

def load_module(name, path):
    spec = importlib.util.spec_from_file_location(name, path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f'Cannot load module: {path}')
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module

def verify_files(directory, hashes):
    for name, expected in hashes.items():
        if file_digest(directory / name) != expected:
            raise RuntimeError(f'Frozen artifact changed: {directory / name}')
print('Read-only prerequisites on CPU; DMD inference device selected below.')
MATCHER_ROOT = PROJECT_ROOT / 'experiments' / 'risk_patch_matcher_v1'
MATCHER_SETTINGS = {'epochs': 6, 'steps_per_epoch': 200, 'identities_per_batch': 12,
    'learning_rate': 0.0002, 'embedding_dim': 128, 'held_groups_per_db': 8,
    'quality_gate': {'clean_auc': 0.80, 'dry_auc': 0.70, 'clean_tar': 0.25, 'dry_tar': 0.10}}

import ast
import io
import sys
import subprocess
import urllib.request
import zipfile
from scipy.optimize import linear_sum_assignment
import scipy
import torch.nn.functional as F
INFERENCE_DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('DMD inference device:', INFERENCE_DEVICE, '| gradient check: CPU only')


In [ ]:
split_protocol = json.loads((SPLIT_DIR / 'protocol.json').read_text(encoding='utf-8'))
verify_files(SPLIT_DIR, split_protocol['file_sha256'])
locked = pd.read_csv(SPLIT_DIR / 'manifest_splits.csv', keep_default_na=False)
m0_config = json.loads((M0_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_config = json.loads((M1_DIR / 'configuration.json').read_text(encoding='utf-8'))
m1_completion = json.loads((M1_DIR / 'completion.json').read_text(encoding='utf-8'))
sim_config = json.loads((SIM_DIR / 'configuration.json').read_text(encoding='utf-8'))
pair_config = json.loads((PAIR_DIR / 'configuration.json').read_text(encoding='utf-8'))
m0_sha, m1_sha = (file_digest(directory / 'configuration.json') for directory in (M0_DIR, M1_DIR))
verify_files(M0_DIR, m0_config['source_file_sha256'])
verify_files(M1_DIR, m1_config['source_file_sha256'])
verify_files(M1_DIR, m1_completion['file_sha256'])
if (m1_completion['status'] != 'shared_curriculum_control_complete' or m1_completion['configuration_sha256'] != m1_sha
        or m1_config['m0_configuration_sha256'] != m0_sha):
    raise RuntimeError('Frozen completed M1 provenance mismatch.')
if any(config['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json') for config in (m0_config, sim_config, pair_config)):
    raise RuntimeError('Split provenance differs across notebooks.')
if (m0_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or m1_config['simulator_configuration_sha256'] != file_digest(SIM_DIR / 'configuration.json')
        or sim_config['normalization_configuration_sha256'] != file_digest(PAIR_DIR / 'configuration.json')
        or file_digest(SIM_DIR / 'fingerprint_degradation.py') != sim_config['module_sha256']
        or m0_config['simulator_module_sha256'] != sim_config['module_sha256']):
    raise RuntimeError('Frozen simulator/normalization provenance mismatch.')
for config in (m0_config, m1_config):
    if (str(torch.__version__) != config['torch_version'] or str(torch.version.cuda) != config['torch_cuda']
            or np.__version__ != config['numpy_version'] or Image.__version__ != config['pillow_version']):
        raise RuntimeError('Library versions differ from frozen training.')
if Image.__version__ != pair_config['pillow_version'] or Image.__version__ != sim_config['pillow_version'] or np.__version__ != sim_config['numpy_version']:
    raise RuntimeError('Normalization/simulator library versions differ.')
for name in ('restoration_baseline.py', 'train_sources.csv', 'validation_sources.csv'):
    if file_digest(M1_DIR / name) != file_digest(M0_DIR / name):
        raise RuntimeError(f'M1 model/source differs from M0: {name}')
SETTINGS = dict(m1_config['settings'])
EPOCHS = len(m1_config['epoch_conditions'])
if (EPOCHS != sum(SETTINGS['phase_lengths']) or EPOCHS != m0_config['epochs']
        or SETTINGS['base_channels'] != m0_config['base_channels'] or SETTINGS['patch_size'] != m0_config['patch_size']
        or SETTINGS['batch_size'] != m0_config['batch_size'] or SETTINGS['learning_rate'] != m0_config['learning_rate']):
    raise RuntimeError('Budget/architecture/optimizer differ from the frozen baseline.')
train_sources = pd.read_csv(M0_DIR / 'train_sources.csv', keep_default_na=False)
validation_sources = pd.read_csv(M0_DIR / 'validation_sources.csv', keep_default_na=False)
for split_name, frame in (('train', train_sources), ('validation', validation_sources)):
    allowed = locked[locked['split'].eq(split_name)].set_index('relative_path')
    if frame.empty or frame['relative_path'].duplicated().any() or frame.groupby(['dataset', 'db']).ngroups != 6 or set(frame['dataset']) != {'FVC2000', 'FVC2002'}:
        raise RuntimeError(f'Invalid source-only {split_name} data.')
    for record in frame.to_dict('records'):
        relative = record['relative_path']
        if (relative not in allowed.index or allowed.loc[relative, 'sha256'] != record['sha256']
                or allowed.loc[relative, 'identity_key'] != record['identity_key']):
            raise RuntimeError(f'Split/source mismatch: {relative}')
        path = DATA_ROOT / relative
        if split_name == 'train' and (not path.is_file() or file_digest(path) != record['sha256']):
            raise RuntimeError(f'Image missing/changed: {path}. Restore with notebook 02.')
if (set(train_sources['identity_key']) & set(validation_sources['identity_key'])
        or set(train_sources['sha256']) & set(validation_sources['sha256'])
        or not validation_sources.groupby(['dataset', 'db']).size().eq(12).all()):
    raise RuntimeError('Train/validation overlap or changed validation coverage.')
training = load_module('m2_original_model_v1', M0_DIR / 'restoration_baseline.py')
simulator = load_module('m2_original_simulator_v1', SIM_DIR / 'fingerprint_degradation.py')
curriculum_data = load_module('m2_frozen_task_data_v1', M1_DIR / 'curriculum_data.py')
FROZEN_ENGINE_SOURCE = (M1_DIR / 'training_helpers.py.txt').read_bytes().decode('utf-8')
SOURCE_COUNT = len(train_sources)
all_conditions = [('identity', 'none')] + [(family, severity)
    for family in ('dry_like', 'wet_like', 'generic') for severity in ('mild', 'moderate', 'severe')]
print('Verified unchanged sources:', SOURCE_COUNT, 'train /', len(validation_sources), 'validation.')

allowed_train_paths = frozenset(train_sources.relative_path)


In [ ]:

pointer = json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8'))
MATCHER_DIR = Path(pointer['directory'])
if (MATCHER_DIR.parent != MATCHER_ROOT
        or file_digest(MATCHER_DIR / 'configuration.json') != pointer['configuration_sha256']
        or file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']):
    raise RuntimeError('Notebook29 published pointer/integrity mismatch.')
configuration = json.loads((MATCHER_DIR / 'configuration.json').read_text(encoding='utf-8'))
completion = json.loads((MATCHER_DIR / 'completion.json').read_text(encoding='utf-8'))
if (configuration['version'] != 'risk_patch_matcher_v1'
        or configuration['settings'] != MATCHER_SETTINGS or configuration['seed'] != SEED
        or completion['status'] != 'risk_matcher_fit_complete'
        or completion['configuration_sha256'] != pointer['configuration_sha256']
        or completion['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or configuration['selection'] != 'last fixed-budget checkpoint; no gate/validation epoch selection'
        or configuration['train_sources_sha256'] != file_digest(M0_DIR / 'train_sources.csv')
        or configuration['split_protocol_sha256'] != file_digest(SPLIT_DIR / 'protocol.json')
        or configuration['simulator_sha256'] != file_digest(SIM_DIR / 'fingerprint_degradation.py')
        or configuration['patch_size'] != SETTINGS['patch_size']):
    raise RuntimeError('Expected unchanged completed29 fixed-budget matcher configuration.')
verify_files(MATCHER_DIR, completion['file_sha256'])
for key, actual in (('torch_version', str(torch.__version__)), ('torch_cuda', str(torch.version.cuda)),
        ('numpy_version', np.__version__), ('pillow_version', Image.__version__)):
    if configuration[key] != actual:
        raise RuntimeError(f'Matcher library version differs: {key}')
matcher_source = (MATCHER_DIR / 'matcher_source.py').read_bytes().decode('utf-8')
if (hashlib.sha256(matcher_source.encode()).hexdigest() != configuration['model_source_sha256']
        or file_digest(MATCHER_DIR / 'inner_partition.csv') != configuration['partition_sha256']
        or file_digest(MATCHER_DIR / 'checkpoint_helpers.py.txt') != configuration['checkpoint_source_sha256']):
    raise RuntimeError('Frozen matcher source/partition hash differs.')
matcher = types.ModuleType('read_only_failed29_matcher')
matcher.__dict__.update(globals())
exec(compile(matcher_source, str(MATCHER_DIR / 'matcher_source.py'), 'exec'), matcher.__dict__)
partition = pd.read_csv(MATCHER_DIR / 'inner_partition.csv', keep_default_na=False)
expected_partition = matcher.inner_partition(train_sources, MATCHER_SETTINGS['held_groups_per_db'])
pd.testing.assert_frame_equal(partition, expected_partition, check_dtype=False)
latest = json.loads((MATCHER_DIR / 'latest.json').read_text(encoding='utf-8'))
if (latest != completion['checkpoint'] or Path(latest['file']).name != latest['file']
        or latest['configuration_sha256'] != pointer['configuration_sha256']
        or latest['completed_epochs'] != MATCHER_SETTINGS['epochs']
        or file_digest(MATCHER_DIR / latest['file']) != latest['sha256']):
    raise RuntimeError('LAST completed29 checkpoint differs.')
state = torch.load(MATCHER_DIR / latest['file'], map_location='cpu', weights_only=True)
if (state['configuration_sha256'] != pointer['configuration_sha256']
        or state['completed_epochs'] != MATCHER_SETTINGS['epochs']):
    raise RuntimeError('Saved matcher payload differs.')
with torch.random.fork_rng(devices=[]):
    model = matcher.PatchMatcher().to(DEVICE)
model.load_state_dict(state['model'], strict=True)
model.eval()
for parameter in model.parameters():
    parameter.requires_grad_(False)
if any(not torch.isfinite(value).all() for value in model.state_dict().values()):
    raise RuntimeError('Nonfinite frozen matcher state.')
saved_state = {key: value.clone() for key, value in model.state_dict().items()}
saved_history = pd.DataFrame(state['history'])
if saved_history.epoch.tolist() != list(range(1, MATCHER_SETTINGS['epochs'] + 1)):
    raise RuntimeError('Matcher has not trained the full predefined budget.')
del state
calibration = pd.read_csv(MATCHER_DIR / 'calibration_scores.csv', float_precision='round_trip')
gate = pd.read_csv(MATCHER_DIR / 'gate_scores.csv', float_precision='round_trip')
saved_gate = json.loads((MATCHER_DIR / 'quality_gate.json').read_text(encoding='utf-8'))
saved_thresholds = json.loads((MATCHER_DIR / 'thresholds.json').read_text(encoding='utf-8'))
if saved_gate != completion['quality_gate']:
    raise RuntimeError('Saved29 gate attestation differs.')
print('Read-only loaded matcher epoch:', latest['completed_epochs'], '| original gate passed:', saved_gate['passed'])


In [ ]:
AUDIT_SOURCE = 'def panel_records(partition, split):\n    if split not in (\'fit\', \'gate\', \'calibration\'):\n        raise ValueError(\'Only explicit main-TRAIN inner partitions are supported.\')\n    selected = partition[partition.matcher_split.eq(split)]\n    if selected.empty:\n        raise RuntimeError(\'Missing inner partition.\')\n    if split == \'fit\':\n        frames = []\n        for (dataset, db), rows in selected.groupby([\'dataset\', \'db\'], sort=True):\n            groups = sorted(rows.connected_group.unique(),\n                key=lambda key: training.stable_seed(f\'{SEED}|audit_fit_panel|{key}\'))\n            if len(groups) < MATCHER_SETTINGS[\'held_groups_per_db\']:\n                raise RuntimeError(\'Insufficient fit groups for matched audit panel.\')\n            frames.append(rows[rows.connected_group.isin(groups[:MATCHER_SETTINGS[\'held_groups_per_db\']])])\n        selected = pd.concat(frames, ignore_index=True)\n    galleries, probes = [], []\n    for key, rows in selected.groupby(\'identity_key\', sort=True):\n        records = rows.sort_values(\'relative_path\').drop_duplicates(\'sha256\').to_dict(\'records\')\n        if len(records) < 3:\n            raise RuntimeError(f\'Audit requires the same three byte-distinct source impressions: {key}\')\n        galleries.append(records[0])\n        probes.extend(records[1:3])\n    return galleries, probes\n\ndef pair_metadata(partition, split):\n    galleries, probes = panel_records(partition, split)\n    rows = []\n    for condition in (\'identity/none\', \'dry_like/mild\', \'dry_like/moderate\', \'dry_like/severe\'):\n        for probe in probes:\n            for gallery in galleries:\n                if (probe[\'dataset\'], probe[\'db\']) != (gallery[\'dataset\'], gallery[\'db\']):\n                    continue\n                label = int(probe[\'identity_key\'] == gallery[\'identity_key\'])\n                if not label and probe[\'connected_group\'] == gallery[\'connected_group\']:\n                    continue\n                rows.append(dict(dataset=probe[\'dataset\'], db=probe[\'db\'], condition=condition,\n                    probe=probe[\'relative_path\'], gallery=gallery[\'relative_path\'], label=label))\n    return pd.DataFrame(rows)\n\ndef verify_score_metadata(frame, partition, split):\n    keys = [\'dataset\', \'db\', \'condition\', \'probe\', \'gallery\', \'label\']\n    if (frame.empty or frame.duplicated(keys).any() or not np.isfinite(frame.score).all()\n            or not frame.score.between(-1.00001, 1.00001).all() or set(frame.label) != {0, 1}):\n        raise RuntimeError(\'Invalid saved inner verification scores.\')\n    expected = pair_metadata(partition, split)\n    pd.testing.assert_frame_equal(frame[keys].sort_values(keys).reset_index(drop=True),\n        expected[keys].sort_values(keys).reset_index(drop=True), check_dtype=False)\n\ndef crop_details(record):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'Crop audit can read main TRAIN images only.\')\n    dpi = pair_config[\'source_dpi\'][f"{record[\'dataset\']}_{record[\'db\']}"]\n    reference = training.normalized_image(str(DATA_ROOT / relative), dpi)\n    foreground = simulator.foreground_mask(reference)\n    ys, xs = np.where(foreground)\n    if not len(ys):\n        raise RuntimeError(f\'No foreground for frozen crop: {relative}\')\n    crop_seed = training.stable_seed(f\'{SEED}|0|{relative}|crop\')\n    rng = np.random.default_rng(crop_seed)\n    anchor = int(rng.integers(len(ys)))\n    size = SETTINGS[\'patch_size\']\n    height, width = reference.shape\n    top = int(np.clip(int(ys[anchor]) - size // 2, 0, max(0, height - size)))\n    left = int(np.clip(int(xs[anchor]) - size // 2, 0, max(0, width - size)))\n    pad = ((0, max(0, size - height)), (0, max(0, size - width)))\n    padded = np.pad(reference, pad, mode=\'reflect\') if any(value for pair in pad for value in pair) else reference\n    patch = np.array(padded[top:top + size, left:left + size], dtype=np.float32, copy=True)\n    actual = matcher.sample_patches([record], 0, [(\'identity\', \'none\')])[0, 0].numpy()\n    if not np.array_equal(patch, actual):\n        raise RuntimeError(\'Reconstructed crop is not byte-identical to frozen29 sampling.\')\n    coverage = (min(size, height - top) * min(size, width - left)) / (height * width)\n    metadata = dict(relative_path=relative, identity_key=record[\'identity_key\'], dataset=record[\'dataset\'],\n        db=record[\'db\'], crop_seed=crop_seed, normalized_height=height, normalized_width=width,\n        top=top, left=left, patch_size=size, padded_bottom=pad[0][1], padded_right=pad[1][1],\n        crop_image_fraction=coverage, crop_foreground_fraction=float(\n            foreground[top:min(top + size, height), left:min(left + size, width)].mean()),\n        crop_sha256=hashlib.sha256(patch.astype(\'<f4\').tobytes()).hexdigest())\n    return metadata, reference, patch\n\ndef describe_scores(values):\n    values = np.asarray(values, dtype=np.float64)\n    if not len(values) or not np.isfinite(values).all():\n        raise RuntimeError(\'Nonempty finite scores required for descriptive summaries.\')\n    return dict(count=len(values), mean=float(values.mean()), std=float(values.std(ddof=1)) if len(values) > 1 else None,\n        q05=float(np.quantile(values, .05)), median=float(np.median(values)), q95=float(np.quantile(values, .95)))\n\ndef auc_against(genuine, impostors):\n    genuine, impostors = np.asarray(genuine), np.asarray(impostors)\n    if not len(genuine) or not len(impostors):\n        raise RuntimeError(\'Both positive and negative scores required.\')\n    return float(np.mean((genuine[:, None] > impostors).astype(float)\n        + .5 * (genuine[:, None] == impostors)))\n\n@torch.inference_mode()\ndef embed_panel(records, condition):\n    features = []\n    for start in range(0, len(records), 12):\n        images = matcher.sample_patches(records[start:start + 12], 0, [condition]).to(DEVICE)\n        value = model(images)\n        if not torch.isfinite(value).all():\n            raise RuntimeError(\'Nonfinite CPU embeddings.\')\n        features.append(value.cpu())\n    return torch.cat(features)\n\ndef summarize_panel(frame):\n    summaries = []\n    for (split, dataset, db, condition), group in frame.groupby(\n            [\'inner_split\', \'dataset\', \'db\', \'condition\'], sort=True):\n        threshold = saved_thresholds[f\'{dataset}_{db}\']\n        positive = group[group.pool.eq(\'different_impression_genuine\')].score.to_numpy()\n        for pool in (\'different_impression_genuine\', \'same_source_control\', \'same_db_impostor\', \'cross_db_impostor\'):\n            values = group[group.pool.eq(pool)].score.to_numpy()\n            summaries.append(dict(inner_split=split, dataset=dataset, db=db, condition=condition, pool=pool,\n                threshold=threshold, acceptance_fraction=float(np.mean(values >= threshold)),\n                auc_against_cross_impression_genuine=auc_against(positive, values)\n                    if pool.endswith(\'impostor\') else None, **describe_scores(values)))\n    return pd.DataFrame(summaries)\n'
exec(compile(AUDIT_SOURCE, '<matcher-failure-audit>', 'exec'))


In [ ]:

verify_score_metadata(calibration, partition, 'calibration')
verify_score_metadata(gate, partition, 'gate')
thresholds, replay_metrics, replay_summary = matcher.gate_analysis(calibration, gate)
if saved_thresholds != {f'{dataset}_{db}': value for (dataset, db), value in thresholds.items()}:
    raise RuntimeError('Exact CSV replay changed original calibration thresholds.')
if replay_summary != saved_gate:
    raise RuntimeError('Saved gate summary differs from exact saved-score replay.')
prior_metrics = pd.read_csv(MATCHER_DIR / 'gate_metrics.csv', float_precision='round_trip')
pd.testing.assert_frame_equal(replay_metrics, prior_metrics, check_dtype=False, check_exact=True)
print('Exact saved-score replay verified; gate UNCHANGED:')
print(json.dumps(replay_summary, indent=2))
print('Saved fixed-budget learning/runtime/memory history:')
print(saved_history.to_string(index=False))


In [ ]:

DMD_COMMIT = 'feb0ad40e5f340caebb457237d7ef21d5d5724e6'
DMD_FILE_HASHES = {'models/model_zoo.py': '39f24735c03bcbbc0d51f6af75146bd3beb55a17bc25cf1a3b1c8f99bdd54b29', 'models/resnet.py': 'e6d28ed8a9730fc0ce5f1a98d464a129cd976ad91028e4516880908b42688e8d', 'models/units.py': '3c685be67e6ad57c5016cf44f75aa7660ba4675f17277faec470d44b3215a1aa', 'models/inception.py': 'bac60ed9f06c9e2e0869f73bb2bf987bbd3889856e4509f9a0cec5070d05702d', 'models/dataloader_densemnt.py': '70308d0b9a6258f5fb2eb815fa296a987190c0db8d9e5750c00c9df16e9468aa', 'evaluate_mnt.py': '84e7f4e62ebdaf718bc1266367cee5fa7680fd599d585c3cc9398f35a4e5b78f', 'README.md': '2b538fe64ef3510f60512a7b413eb04bbfc66a4c51b4fa00aa67e533eb84ee68', 'LICENCE': '4878c4e24ae4dd9878852724b4ae3313dc9a0599680d6552c1606bbc3d63830c', 'DMD.yaml': '12186bc0a33ab479d9c041aae8e35a180a286b98500c3d990475403732538b3d'}
DMD_WEIGHT_SHA = '6d89dff9e5502cf4534b0fd9e014b3fb4a494a0be904015319c72edd9a7aff48'
DMD_WEIGHT_BYTES = 179093287
DMD_WEIGHT_URL = 'https://drive.usercontent.google.com/download?id=1iu8pbI2EjSkwOYsYczosJJDEhd1-JyzU&export=download&confirm=t'
ASSET_ROOT = PROJECT_ROOT / 'pretrained' / 'dmd_sd14_v1' / DMD_COMMIT
ASSET_ROOT.mkdir(parents=True, exist_ok=True)

def download_checked(url, path, sha, size=None):
    if path.exists():
        if file_digest(path) != sha or (size is not None and path.stat().st_size != size):
            raise RuntimeError(f'Cached public asset failed integrity: {path}')
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    pending = path.with_suffix(path.suffix + '.part')
    with urllib.request.urlopen(url, timeout=180) as response, pending.open('wb') as stream:
        while True:
            chunk = response.read(8 * 1024 * 1024)
            if not chunk:
                break
            stream.write(chunk)
    actual_sha = file_digest(pending)
    actual_size = pending.stat().st_size
    if actual_sha != sha or (size is not None and actual_size != size):
        raise RuntimeError(f'Public download failed SHA256/size: expected_sha256={sha}, '
            f'actual_sha256={actual_sha}, expected_bytes={size}, actual_bytes={actual_size}; '
            f'retained partial: {pending}')
    pending.replace(path)

for name, sha in DMD_FILE_HASHES.items():
    download_checked(f'https://raw.githubusercontent.com/Yu-Yy/DMD/{DMD_COMMIT}/{name}',
        ASSET_ROOT / name, sha)
WEIGHT_PATH = ASSET_ROOT / 'best_model.pth.tar'
download_checked(DMD_WEIGHT_URL, WEIGHT_PATH, DMD_WEIGHT_SHA, DMD_WEIGHT_BYTES)
NBIS_ROOT = LOCAL_ROOT / 'tools' / 'nbis_5_0_0'
MINDTCT = NBIS_ROOT / 'install' / 'bin' / 'mindtct'
build = json.loads((NBIS_ROOT / 'verified_build.json').read_text(encoding='utf-8'))
if (build['signature']['source_sha256'] != pair_config['nbis_source_sha256']
        or pair_config['nbis_source_sha256'] != '0adf8ab0f6b0e4208de50ca00ba21d3d77112ecd66288757ddfed21f6bee92c3'
        or not MINDTCT.is_file() or file_digest(MINDTCT) != build['binary_sha256']['mindtct']
        or pair_config['mindtct_flags'] != []):
    raise RuntimeError('Run04 first: expected unchanged official NBIS5 mindtct with no flags.')
nbis_archive = PROJECT_ROOT / 'archives' / 'nbis_v5_0_0.zip'
if file_digest(nbis_archive) != pair_config['nbis_source_sha256']:
    raise RuntimeError('NBIS source archive changed.')
with zipfile.ZipFile(nbis_archive) as archive:
    conversion_source = archive.read('Rel_5.0.0/mindtct/src/lib/mindtct/xytreps.c')
if hashlib.sha256(conversion_source).hexdigest() != '93a79128b210b9838660f45cdcf3da0fb82c91ae2875da7847d2e9854854de50':
    raise RuntimeError('NBIS XYT convention source changed.')
print('Verified official pinned DMD source,179093287-byte safe-load candidate and NBIS conversion source.')


In [ ]:
ADAPTER_SOURCE = 'def load_dmd(source_dir, weight_path):\n    namespace = \'_frozen_dmd33\'\n    package = types.ModuleType(namespace)\n    package.__path__ = []\n    sys.modules[namespace] = package\n    for name in (\'inception\', \'units\', \'resnet\', \'model_zoo\'):\n        full_name = f\'{namespace}.{name}\'\n        module = types.ModuleType(full_name)\n        module.__package__ = namespace\n        sys.modules[full_name] = module\n        setattr(package, name, module)\n        path = source_dir / \'models\' / f\'{name}.py\'\n        exec(compile(path.read_bytes().decode(\'utf-8\'), str(path), \'exec\'), module.__dict__)\n    with torch.random.fork_rng(devices=[]):\n        result = package.model_zoo.DMD(ndim_feat=6, pos_embed=True, tar_shape=(128, 128), input_norm=False)\n    payload = torch.load(weight_path, map_location=\'cpu\', weights_only=True)\n    if not isinstance(payload, dict) or set(payload) != {\'model\', \'epoch\'} or payload[\'epoch\'] != 126:\n        raise RuntimeError(\'Unexpected public DMD payload; no unsafe fallback permitted.\')\n    result.load_state_dict(payload[\'model\'], strict=True)\n    result.eval()\n    for parameter in result.parameters():\n        parameter.requires_grad_(False)\n    if any(not torch.isfinite(value).all() for value in result.state_dict().values()):\n        raise RuntimeError(\'Nonfinite pretrained DMD state.\')\n    return result\n\ndef state_sha(module):\n    digest = hashlib.sha256()\n    for key, value in module.state_dict().items():\n        digest.update(key.encode())\n        digest.update(value.detach().cpu().contiguous().numpy().tobytes())\n    return digest.hexdigest()\n\ndef cpu_assignment(cost):\n    if cost.ndim != 3 or cost.shape[1] != cost.shape[2] or not torch.isfinite(cost).all():\n        raise RuntimeError(\'Finite square batched assignment costs required.\')\n    rows = []\n    for matrix in cost.detach().cpu().numpy():\n        row, col = linear_sum_assignment(matrix)\n        if not np.array_equal(row, np.arange(len(matrix))):\n            raise RuntimeError(\'Unexpected assignment ordering.\')\n        rows.append(col)\n    return torch.as_tensor(np.stack(rows), device=cost.device, dtype=torch.long)\n\ndef load_scoring(source_path):\n    tree = ast.parse(source_path.read_bytes().decode(\'utf-8\'))\n    names = {\'calculate_score_torchB\', \'lsar_score_torchB\'}\n    selected = [node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name in names]\n    if {node.name for node in selected} != names:\n        raise RuntimeError(\'Upstream scoring definitions differ.\')\n    scope = dict(torch=torch, batch_linear_assignment=cpu_assignment)\n    exec(compile(ast.Module(body=selected, type_ignores=[]), str(source_path), \'exec\'), scope)\n    return scope[\'calculate_score_torchB\'], scope[\'lsar_score_torchB\']\n\ndef parse_nbis_xyt(text, height, width, maximum=24):\n    rows = []\n    for line in text.splitlines():\n        if not line.strip():\n            continue\n        values = [int(value) for value in line.split()]\n        if len(values) != 4:\n            raise RuntimeError(\'Expected four integer NBIS XYT columns.\')\n        x, y, theta, quality = values\n        if not (0 <= x < width and 1 <= y <= height and 0 <= theta < 360 and 0 <= quality <= 100):\n            raise RuntimeError(\'Out-of-range NBIS XYT values.\')\n        rows.append((x, height - y, (-theta) % 360, quality))\n    unique = sorted(set(rows), key=lambda row: (-row[3], row[0], row[1], row[2]))\n    seen, selected = set(), []\n    for row in unique:\n        if row[:3] not in seen:\n            selected.append(row)\n            seen.add(row[:3])\n    return np.asarray(selected[:maximum], dtype=np.float32).reshape(-1, 4)\n\ndef aligned_patches(image, anchors):\n    if image.ndim != 4 or image.shape[:2] != (1, 1) or not torch.isfinite(image).all():\n        raise RuntimeError(\'Expected one finite N1HW source image.\')\n    points = torch.as_tensor(anchors, device=image.device, dtype=image.dtype)\n    if points.ndim != 2 or points.shape[1] < 3 or not len(points) or not torch.isfinite(points).all():\n        raise RuntimeError(\'Expected nonempty finite minutia anchors.\')\n    height, width = image.shape[-2:]\n    yy, xx = torch.meshgrid(torch.arange(128, device=image.device, dtype=image.dtype) - 64,\n        torch.arange(128, device=image.device, dtype=image.dtype) - 64, indexing=\'ij\')\n    theta = torch.deg2rad(points[:, 2])[:, None, None]\n    sx = points[:, 0, None, None] + torch.cos(theta) * xx - torch.sin(theta) * yy\n    sy = points[:, 1, None, None] + torch.sin(theta) * xx + torch.cos(theta) * yy\n    grid = torch.stack(((2 * sx + 1) / width - 1, (2 * sy + 1) / height - 1), dim=-1)\n    return 2 * F.grid_sample((image - .5).expand(len(points), -1, -1, -1), grid,\n        mode=\'bilinear\', padding_mode=\'zeros\', align_corners=False)\n\ndef descriptor_batch(module, image, anchors, batch_size=8):\n    features, masks = [], []\n    for start in range(0, len(anchors), batch_size):\n        result = module.get_embedding(aligned_patches(image, anchors[start:start + batch_size]))\n        features.append(result[\'feature\'])\n        masks.append(result[\'mask\'])\n    feature, mask = torch.cat(features), torch.cat(masks)\n    if (feature.shape != (len(anchors), 768) or mask.shape != (len(anchors), 64)\n            or not torch.isfinite(feature).all() or not torch.isfinite(mask).all()\n            or not ((mask >= 0) & (mask <= 1)).all()):\n        raise RuntimeError(\'Invalid pretrained DMD output shapes/values.\')\n    return feature, mask\n\ndef dense_scores(first, second):\n    return local_score(first[0][None], second[0][None], first[1][None], second[1][None],\n        ndim_feat=12, Normalize=False, binary=False)[0]\n\ndef fingerprint_score(first, second):\n    matrix = dense_scores(first, second)\n    score = lsar_score(matrix[None], first[2][None, :, :3], second[2][None, :, :3])[0]\n    if not torch.isfinite(score):\n        raise RuntimeError(\'Nonfinite official LSA-R score.\')\n    return float(score)\n\ndef original_train_image(record):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'DMD feasibility may open main TRAIN images only.\')\n    path = DATA_ROOT / relative\n    if file_digest(path) != record[\'sha256\']:\n        raise RuntimeError(f\'TRAIN source changed: {relative}\')\n    dpi = pair_config[\'source_dpi\'][f"{record[\'dataset\']}_{record[\'db\']}"]\n    return training.normalized_image(str(path), dpi)\n\ndef atomic_npz(path, **arrays):\n    stream = io.BytesIO()\n    np.savez_compressed(stream, **arrays)\n    write_verified(path, stream.getvalue())\n\ndef cached_bundle(directory, expected):\n    marker = directory / \'completion.json\'\n    if not marker.exists():\n        return None\n    saved = json.loads(marker.read_text(encoding=\'utf-8\'))\n    if saved[\'configuration\'] != expected:\n        raise RuntimeError(f\'Cached image/feature provenance differs: {directory}\')\n    verify_files(directory, saved[\'file_sha256\'])\n    return saved\n\ndef seal_bundle(directory, expected, names):\n    write_verified(directory / \'completion.json\', json_bytes(dict(configuration=expected,\n        file_sha256={name: file_digest(directory / name) for name in names})))\n\ndef image_anchors(record):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths:\n        raise RuntimeError(\'Anchor extraction is main TRAIN only.\')\n    key = hashlib.sha256(relative.encode()).hexdigest()[:24]\n    directory = FEASIBILITY_DIR / \'anchors\' / key\n    directory.mkdir(parents=True, exist_ok=True)\n    expected = dict(relative_path=relative, source_sha256=record[\'sha256\'],\n        configuration_sha256=feasibility_sha, mindtct_sha256=build[\'binary_sha256\'][\'mindtct\'])\n    saved = cached_bundle(directory, expected)\n    if saved is None:\n        reference = original_train_image(record)\n        image_path = directory / \'original.png\'\n        Image.fromarray(np.rint(reference * 255).astype(np.uint8)).save(image_path, dpi=(500, 500))\n        output = directory / \'minutiae\'\n        result = subprocess.run([str(MINDTCT), str(image_path), str(output)], capture_output=True,\n            text=True, timeout=120)\n        write_verified(directory / \'mindtct.log\', (result.stdout + result.stderr).encode())\n        if result.returncode or not output.with_suffix(\'.xyt\').is_file():\n            raise RuntimeError(f\'MINDTCT failed for {relative}; see {directory / "mindtct.log"}\')\n        anchors = parse_nbis_xyt(output.with_suffix(\'.xyt\').read_text(encoding=\'ascii\'), *reference.shape)\n        atomic_npz(directory / \'anchors.npz\', anchors=anchors, image_shape=np.array(reference.shape))\n        seal_bundle(directory, expected, [\'original.png\', \'minutiae.xyt\', \'mindtct.log\', \'anchors.npz\'])\n    with np.load(directory / \'anchors.npz\', allow_pickle=False) as archive:\n        anchors = archive[\'anchors\']\n        shape = archive[\'image_shape\']\n    text = (directory / \'minutiae.xyt\').read_text(encoding=\'ascii\')\n    expected_anchors = parse_nbis_xyt(text, *shape)\n    if not np.array_equal(anchors, expected_anchors):\n        raise RuntimeError(\'Cached anchors disagree with their original NBIS XYT.\')\n    return anchors\n\ndef condition_image(record, family, severity):\n    reference = original_train_image(record)\n    if family == \'identity\':\n        return reference\n    seed = simulator.case_seed(record[\'relative_path\'], family, severity, SEED)\n    degraded, _ = simulator.degrade(reference, family, severity, seed)\n    return degraded\n\n@torch.inference_mode()\ndef image_features(record, family, severity, anchors):\n    relative = record[\'relative_path\']\n    if relative not in allowed_train_paths or len(anchors) < 4:\n        raise RuntimeError(\'Expected a matchable TRAIN record; missing anchors must be reported upstream.\')\n    key = hashlib.sha256(f\'{relative}|{family}|{severity}\'.encode()).hexdigest()[:24]\n    directory = FEASIBILITY_DIR / \'features\' / key\n    directory.mkdir(parents=True, exist_ok=True)\n    expected = dict(relative_path=relative, family=family, severity=severity,\n        configuration_sha256=feasibility_sha, anchors_sha256=hashlib.sha256(anchors.tobytes()).hexdigest())\n    if cached_bundle(directory, expected) is None:\n        array = condition_image(record, family, severity)\n        image = torch.from_numpy(array.copy())[None, None].to(INFERENCE_DEVICE)\n        feature, mask = descriptor_batch(dmd_model, image, anchors)\n        atomic_npz(directory / \'features.npz\', feature=feature.cpu().numpy(), mask=mask.cpu().numpy())\n        seal_bundle(directory, expected, [\'features.npz\'])\n    with np.load(directory / \'features.npz\', allow_pickle=False) as archive:\n        feature = torch.from_numpy(archive[\'feature\'].copy())\n        mask = torch.from_numpy(archive[\'mask\'].copy())\n    if (feature.shape != (len(anchors), 768) or mask.shape != (len(anchors), 64)\n            or not torch.isfinite(feature).all() or not torch.isfinite(mask).all()\n            or not ((mask >= 0) & (mask <= 1)).all()):\n        raise RuntimeError(\'Invalid cached DMD features.\')\n    return feature, mask, torch.from_numpy(anchors.copy())\n\ndef quality_analysis(calibration_scores, gate_scores):\n    thresholds, rows = {}, []\n    for (dataset, db), group in calibration_scores.groupby([\'dataset\', \'db\'], sort=True):\n        negatives = group[group.condition.eq(\'identity/none\') & group.label.eq(0)].score.dropna().to_numpy()\n        if len(negatives) < 50:\n            thresholds[f\'{dataset}_{db}\'] = None\n            print(f\'UNMATCHABLE calibration {dataset}/{db}: only {len(negatives)} valid impostors; gate fails.\')\n        else:\n            allowed = int(np.floor(.01 * len(negatives) + 1e-12))\n            thresholds[f\'{dataset}_{db}\'] = float(np.nextafter(np.sort(negatives)[::-1][allowed], np.inf))\n    for (dataset, db, condition), group in gate_scores.groupby([\'dataset\', \'db\', \'condition\'], sort=True):\n        threshold = thresholds[f\'{dataset}_{db}\']\n        positives, negatives = (group[group.label.eq(label)] for label in (1, 0))\n        genuine = positives.score.dropna().to_numpy()\n        impostors = negatives.score.dropna().to_numpy()\n        auc = auc_against(genuine, impostors) if len(genuine) and len(impostors) else None\n        rows.append(dict(dataset=dataset, db=db, condition=condition, threshold=threshold, auc=auc,\n            pair_matchability=float(group.score.notna().mean()), genuine_attempts=len(positives),\n            impostor_attempts=len(negatives), genuine_matchable=len(genuine), impostor_matchable=len(impostors),\n            tar_attempted=float(np.sum(genuine >= threshold) / len(positives)) if threshold is not None else 0.0,\n            fmr_conditional=float(np.mean(impostors >= threshold))\n                if threshold is not None and len(impostors) else None))\n    metrics = pd.DataFrame(rows)\n    clean = metrics[metrics.condition.eq(\'identity/none\')]\n    dry = metrics[~metrics.condition.eq(\'identity/none\')]\n    summary = dict(clean_auc=float(clean.auc.mean()) if clean.auc.notna().all() else None,\n        dry_auc=float(dry.auc.mean()) if dry.auc.notna().all() else None,\n        clean_tar=float(clean.tar_attempted.mean()), dry_tar=float(dry.tar_attempted.mean()),\n        minimum_pair_matchability=float(metrics.pair_matchability.min()),\n        limits=dict(clean_auc=.8, dry_auc=.7, clean_tar=.25, dry_tar=.1, minimum_pair_matchability=.9))\n    summary[\'passed\'] = bool(all(summary[key] is not None and summary[key] >= minimum\n        for key, minimum in summary[\'limits\'].items()) and all(value is not None for value in thresholds.values()))\n    summary[\'permits_restoration_training\'] = False\n    summary[\'interpretation\'] = \'Development feasibility only; conditional AUC/FMR, attempted-pair TAR. Not independent evaluation or a differentiable full matcher.\'\n    return thresholds, metrics, summary\n'
exec(compile(ADAPTER_SOURCE, '<frozen-dmd33-adapter>', 'exec'))


In [ ]:

implementation_sha = '3eacc029995f4775d564bc7195ff068105831f9a1cb12c4d89384d92d9e906b5'
configuration33 = dict(version='pretrained_dmd_feasibility_v1', implementation_sha256=implementation_sha,
    adapter_sha256=hashlib.sha256(ADAPTER_SOURCE.encode()).hexdigest(), source_commit=DMD_COMMIT,
    source_files=DMD_FILE_HASHES, weight_sha256=DMD_WEIGHT_SHA, weight_bytes=DMD_WEIGHT_BYTES,
    matcher29_configuration_sha256=pointer['configuration_sha256'],
    matcher29_completion_sha256=pointer['completion_sha256'], nbis_build=build,
    dpi=500, patch_size=128, max_anchors=24, minimum_anchors=4,
    scoring='official masked dense cosine; no score normalization; official LSA-R with SciPy Hungarian',
    clean_calibration_fmr=.01, quality_limits=dict(clean_auc=.8,dry_auc=.7,clean_tar=.25,dry_tar=.1,minimum_pair_matchability=.9),
    anchor_policy='original only; quality descending then x/y/theta; fixed across degradations',
    torch_version=str(torch.__version__), torch_cuda=str(torch.version.cuda), numpy_version=np.__version__,
    pillow_version=Image.__version__, scipy_version=scipy.__version__, inference_device=str(INFERENCE_DEVICE),
    cuda_name=torch.cuda.get_device_name() if INFERENCE_DEVICE.type=='cuda' else None,
    cuda_capability=torch.cuda.get_device_capability() if INFERENCE_DEVICE.type=='cuda' else None,
    cpu_threads=torch.get_num_threads(), training=False, permits_restoration_training=False)
feasibility_sha = hashlib.sha256(json_bytes(configuration33)).hexdigest()
FEASIBILITY_DIR = PROJECT_ROOT / 'diagnostics' / 'pretrained_dmd_feasibility_v1' / feasibility_sha[:20]
FEASIBILITY_DIR.mkdir(parents=True, exist_ok=True)
for name, data in {'configuration.json': json_bytes(configuration33),
        'adapter.py.txt': ADAPTER_SOURCE.encode()}.items():
    path = FEASIBILITY_DIR / name
    if path.exists() and path.read_bytes() != data:
        raise RuntimeError(f'Frozen DMD feasibility configuration changed: {path}')
    if not path.exists():
        write_verified(path, data)
local_score, lsar_score = load_scoring(ASSET_ROOT / 'evaluate_mnt.py')
dmd_model = load_dmd(ASSET_ROOT, WEIGHT_PATH)
initial_dmd_sha = state_sha(dmd_model)
if sum(parameter.numel() for parameter in dmd_model.parameters()) != 44636115:
    raise RuntimeError('Unexpected DMD model size.')
print('Strict public weights-only load passed: epoch126,44,636,115 frozen parameters.')


In [ ]:

# Cardinal and subpixel tests compare to an independent scalar bilinear sampler.
ramp = torch.arange(35, dtype=torch.float32).reshape(1,1,5,7) / 34
for theta in (0, 90, 180, 270, 37):
    point = np.array([[3.25, 2.125, theta]], dtype=np.float32)
    patch = aligned_patches(ramp, point)[0,0]
    angle = np.deg2rad(theta)
    for oy, ox in ((64,64),(64,65),(65,64),(0,0),(63,66)):
        sx = 3.25 + np.cos(angle)*(ox-64) - np.sin(angle)*(oy-64)
        sy = 2.125 + np.sin(angle)*(ox-64) + np.cos(angle)*(oy-64)
        x0,y0 = math.floor(sx),math.floor(sy)
        expected = 0.0
        for dy in (0,1):
            for dx in (0,1):
                x,y = x0+dx,y0+dy
                pixel = float(ramp[0,0,y,x]) if 0 <= x < 7 and 0 <= y < 5 else .5
                expected += pixel*(1-abs(sx-x))*(1-abs(sy-y))
        if abs(float(patch[oy,ox]) - (2*expected-1)) > 2e-5:
            raise RuntimeError('Tensor crop disagrees with zero-flow affine mapping.')
converted = parse_nbis_xyt('2 8 0 90\n3 7 90 80\n4 6 180 70\n5 5 270 60', 10, 10)
np.testing.assert_array_equal(converted[:,:3], [[2,2,0],[3,3,270],[4,4,180],[5,5,90]])
print('Geometry verified: NBIS release conversion, four cardinal directions, subpixel and border sampling.')


In [ ]:

panels, selected = {}, {}
for split in ('calibration', 'gate'):
    galleries, probes = panel_records(partition, split)
    panels[split] = (galleries, probes)
    for record in galleries + probes:
        selected[record['relative_path']] = record
anchor_map, anchor_rows = {}, []
for index, (relative, record) in enumerate(sorted(selected.items()), 1):
    anchors = image_anchors(record)
    anchor_map[relative] = anchors
    status = 'matchable' if len(anchors) >= 4 else 'unmatchable_fewer_than4_original_minutiae'
    anchor_rows.append(dict(relative_path=relative, identity_key=record['identity_key'],
        dataset=record['dataset'], db=record['db'], anchors=len(anchors), status=status))
    if len(anchors) < 4:
        print(f'UNMATCHABLE: {relative}: {len(anchors)} original minutiae; no synthetic zero score.')
    if index % 12 == 0:
        print(f'Original TRAIN anchors {index}/{len(selected)}', flush=True)
write_csv(FEASIBILITY_DIR / 'anchor_summary.csv', pd.DataFrame(anchor_rows))


In [ ]:

gradient_record = next((record for record in panels['calibration'][0]
    if len(anchor_map[record['relative_path']]) >= 4), None)
if gradient_record is None:
    raise RuntimeError('No matchable calibration original for input-gradient check; see anchor_summary.csv.')
points = anchor_map[gradient_record['relative_path']][:2]
original = torch.from_numpy(original_train_image(gradient_record).copy())[None,None]
degraded = torch.from_numpy(condition_image(gradient_record, 'dry_like', 'moderate').copy())[None,None].requires_grad_()
with torch.no_grad():
    target_feature, target_mask = descriptor_batch(dmd_model, original, points, batch_size=2)
probe_feature, probe_mask = descriptor_batch(dmd_model, degraded, points, batch_size=2)
local = dense_scores((probe_feature, probe_mask), (target_feature, target_mask)).diagonal().mean()
input_gradient, = torch.autograd.grad(local, degraded)
gradient_report = dict(device='cpu', relative_path=gradient_record['relative_path'],
    score=float(local.detach()), finite=bool(torch.isfinite(input_gradient).all()),
    absolute_sum=float(input_gradient.abs().sum()), absolute_max=float(input_gradient.abs().max()),
    frozen_model_unchanged=state_sha(dmd_model) == initial_dmd_sha,
    parameter_gradients_absent=all(parameter.grad is None for parameter in dmd_model.parameters()),
    scope='fixed anchors -> tensor crop -> pretrained descriptor -> local dense cosine, NOT complete LSA-R')
write_verified(FEASIBILITY_DIR / 'input_gradient.json', json_bytes(gradient_report))
if (not gradient_report['finite'] or gradient_report['absolute_sum'] <= 0
        or not gradient_report['frozen_model_unchanged'] or not gradient_report['parameter_gradients_absent']):
    raise RuntimeError('Pretrained local input-gradient check failed; diagnostics saved.')
print(json.dumps(gradient_report, indent=2))
del probe_feature, probe_mask, target_feature, target_mask, input_gradient, degraded, original, local
dmd_model.to(INFERENCE_DEVICE)
if INFERENCE_DEVICE.type == 'cuda':
    torch.cuda.reset_peak_memory_stats()


In [ ]:

started = time.monotonic()
score_frames, control_rows = {}, []
for split, (galleries, probes) in panels.items():
    feature_bank = {}
    for record in galleries + probes:
        relative = record['relative_path']
        anchors = anchor_map[relative]
        if len(anchors) >= 4:
            feature_bank[(relative,'identity','none')] = image_features(record,'identity','none',anchors)
    rows = []
    for family,severity in [('identity','none')] + [('dry_like',level) for level in ('mild','moderate','severe')]:
        for index, probe in enumerate(probes):
            relative = probe['relative_path']
            key = (relative,family,severity)
            if key not in feature_bank and len(anchor_map[relative]) >= 4:
                feature_bank[key] = image_features(probe,family,severity,anchor_map[relative])
            if key in feature_bank:
                same = dense_scores(feature_bank[key], feature_bank[(relative,'identity','none')]).diagonal()
                control_rows.append(dict(inner_split=split,probe=relative,condition=f'{family}/{severity}',
                    mean_same_anchor_dense_score=float(same.mean()),anchors=len(same)))
            for gallery in galleries:
                if (probe['dataset'],probe['db']) != (gallery['dataset'],gallery['db']):
                    continue
                label = int(probe['identity_key'] == gallery['identity_key'])
                if not label and probe['connected_group'] == gallery['connected_group']:
                    continue
                gallery_key = (gallery['relative_path'],'identity','none')
                matchable = key in feature_bank and gallery_key in feature_bank
                score = fingerprint_score(feature_bank[key],feature_bank[gallery_key]) if matchable else None
                rows.append(dict(dataset=probe['dataset'],db=probe['db'],condition=f'{family}/{severity}',
                    probe=relative,gallery=gallery['relative_path'],label=label,score=score,
                    status='matched' if matchable else 'unmatchable_original_anchor_count'))
            if (index+1) % 16 == 0:
                print(f'DMD {split}/{family}/{severity}: {index+1}/{len(probes)} probes',flush=True)
        if family != 'identity':
            for probe in probes:
                feature_bank.pop((probe['relative_path'],family,severity),None)
    scores33 = pd.DataFrame(rows)
    keys = ['dataset','db','condition','probe','gallery','label']
    pd.testing.assert_frame_equal(scores33[keys].sort_values(keys).reset_index(drop=True),
        pair_metadata(partition,split)[keys].sort_values(keys).reset_index(drop=True),check_dtype=False)
    if not np.isfinite(scores33.score.dropna()).all():
        raise RuntimeError('Nonfinite DMD score; missing anchors must have explicit status.')
    score_frames[split] = scores33
    write_csv(FEASIBILITY_DIR / f'{split}_scores.csv',scores33)
write_csv(FEASIBILITY_DIR / 'same_anchor_controls.csv',pd.DataFrame(control_rows))
runtime_report = dict(seconds=time.monotonic()-started,inference_device=str(INFERENCE_DEVICE),
    peak_allocated_bytes=torch.cuda.max_memory_allocated() if INFERENCE_DEVICE.type=='cuda' else None,
    peak_reserved_bytes=torch.cuda.max_memory_reserved() if INFERENCE_DEVICE.type=='cuda' else None,
    note='Includes cache hits if rerun; not a controlled benchmark.')


In [ ]:

thresholds33, metrics33, summary33 = quality_analysis(score_frames['calibration'],score_frames['gate'])
write_verified(FEASIBILITY_DIR / 'thresholds.json',json_bytes(thresholds33))
write_csv(FEASIBILITY_DIR / 'gate_metrics.csv',metrics33)
write_verified(FEASIBILITY_DIR / 'feasibility.json',json_bytes(summary33))
write_verified(FEASIBILITY_DIR / 'runtime.json',json_bytes(runtime_report))
if state_sha(dmd_model) != initial_dmd_sha or any(parameter.grad is not None for parameter in dmd_model.parameters()):
    raise RuntimeError('Frozen pretrained DMD changed.')
if any(not torch.equal(value,model.state_dict()[key]) for key,value in saved_state.items()):
    raise RuntimeError('Original failed29 model changed.')
verify_files(MATCHER_DIR,completion['file_sha256'])
if file_digest(MATCHER_DIR / 'completion.json') != pointer['completion_sha256']:
    raise RuntimeError('Original29 completion changed.')
if json.loads((MATCHER_ROOT / 'completed_experiment.json').read_text(encoding='utf-8')) != pointer:
    raise RuntimeError('Original29 pointer changed.')
verify_files(ASSET_ROOT,DMD_FILE_HASHES)
if file_digest(WEIGHT_PATH) != DMD_WEIGHT_SHA:
    raise RuntimeError('Public DMD weight file changed.')
names = ('configuration.json','adapter.py.txt','anchor_summary.csv','input_gradient.json',
    'calibration_scores.csv','gate_scores.csv','same_anchor_controls.csv','thresholds.json',
    'gate_metrics.csv','feasibility.json','runtime.json')
write_verified(FEASIBILITY_DIR / 'completion.json',json_bytes(dict(status='pretrained_dmd_feasibility_complete',
    configuration_sha256=feasibility_sha,quality=summary33,permits_restoration_training=False,
    file_sha256={name:file_digest(FEASIBILITY_DIR / name) for name in names})))
print(metrics33.to_string(index=False))
print(json.dumps(summary33,indent=2))
print(json.dumps(runtime_report,indent=2))
print('DMD FEASIBILITY COMPLETE:',FEASIBILITY_DIR)
print('30/31 REMAIN BLOCKED. Send feasibility, gate_metrics, input_gradient and anchor_summary outputs.')
if not summary33['passed']:
    print('DMD FEASIBILITY FAILED predefined development criteria. No training or gate relaxation.')
else:
    print('Development signal criteria passed, NOT an independent benchmark or approval of30/31.')
